In [83]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

## Load dataset

In [84]:
def load_data(file_path):
    try:
        data = pd.read_csv(file_path)
        return data
    except Exception as e:
        print(f"Error loading data: {e}")
        return None
    
df = load_data("../../Database/PDF/Customer-Churn-dataset.csv")

## Fraud logic (Rule-Based)

In [85]:
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"])

# Create fraud logic on condition
import numpy as np
conditions = (
    (df["Balance"] > 150000) &
    (df["NumOfProducts"] <=1 ) &
    (df["IsActiveMember"] == 0)
) | (
    (df["CreditScore"] < 400) &
    (df["Balance"] > 100000)
) | (
    (df["Complain"] > 0) &
    (df["Satisfaction Score"] <= 2)
)

df["Fraud"] = np.where(conditions, 1, 0)

In [86]:
# reducte fraud cases aritifically
fraud_idx = df[df["Fraud"] == 1].sample(frac=0.3, random_state=42).index
df["Fraud"] = 0
df.loc[fraud_idx, "Fraud"] = 1

## Add advanced fraud features

In [87]:
# High risk score (engineered features)
df["RiskScore"] = (
    (df["CreditScore"] < 500).astype(int) +
    (df["Balance"] > 100000).astype(int) +
    (df["IsActiveMember"] == 0).astype(int) +
    (df["Complain"] > 0).astype(int)
)

# Balance per product
df["BalancePerProduct"] = df["Balance"] / (df["NumOfProducts"] + 1)

# Age risk bucket
df["AgeRisk"] = np.where(df["Age"] < 25, 1, np.where(df["Age"] > 60, 1, 0))

In [88]:
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned,Fraud,RiskScore,BalancePerProduct,AgeRisk
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464,0,1,0.000000,0
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456,0,1,41903.930000,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377,0,3,39915.200000,0
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350,0,1,0.000000,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425,0,1,62755.410000,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0,0,1,DIAMOND,300,0,1,0.000000,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0,0,5,PLATINUM,771,0,0,28684.805000,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1,1,3,SILVER,564,0,1,0.000000,0
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1,1,2,GOLD,339,1,2,25025.103333,0


In [89]:
print(df["Fraud"].value_counts(normalize=True))

0    0.9668
1    0.0332
Name: Fraud, dtype: float64


## Add features for Market Risk

In [90]:
df["HighValueCustomer"] = (df["Balance"] > 100000).astype(int)
df["LowCreditRisk"] = (df["CreditScore"] < 500).astype(int)

# Mkarketing score
"""Marketing score: combines high value customer status, low credit risk, and non-France geography to identify customers 
who may be more receptive to marketing campaigns, which can help target efforts effectively."""
df["MarketingScore"] = (
    df["HighValueCustomer"] + df["LowCreditRisk"] + (df["Geography"] != "France").astype(int)
).drop(columns=["HighValueCustomer", "LowCreditRisk"])

## Operation Risk feature for internal banking

In [91]:
df["ComplainFlag"] = (df["Complain"] > 0).astype(int)
df["LowSatisfaction"] = (df["Satisfaction Score"] <= 2).astype(int)

# Systemic risk score
"""Operational risk score: combines complaint and satisfaction flags with inactivity to identify customers at risk of leaving or being dissatisfied, 
which can impact operational stability."""
df["OperationalRiskScore"] = (
    df["ComplainFlag"] +
    df["LowSatisfaction"] +
    (df["IsActiveMember"] == 0).astype(int)
).drop(columns=["ComplainFlag", "LowSatisfaction"])

In [92]:
pd.set_option("display.max_columns", None)
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned,Fraud,RiskScore,BalancePerProduct,AgeRisk,HighValueCustomer,LowCreditRisk,MarketingScore,ComplainFlag,LowSatisfaction,OperationalRiskScore
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1,1,2,DIAMOND,464,0,1,0.000000,0,0,0,0,1,1,2
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0,1,3,DIAMOND,456,0,1,41903.930000,0,0,0,1,1,0,1
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1,1,3,DIAMOND,377,0,3,39915.200000,0,1,0,1,1,0,2
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0,0,5,GOLD,350,0,1,0.000000,0,0,0,0,0,0,1
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0,0,5,GOLD,425,0,1,62755.410000,0,1,0,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0,0,1,DIAMOND,300,0,1,0.000000,0,0,0,0,0,1,2
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0,0,5,PLATINUM,771,0,0,28684.805000,0,0,0,0,0,0,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1,1,3,SILVER,564,0,1,0.000000,0,0,0,0,1,0,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1,1,2,GOLD,339,1,2,25025.103333,0,0,0,1,1,1,3


## Data encoding

In [93]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
# Encode categorical features
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = le.fit_transform(df[col])
    print(f"Encoded {col}")

Encoded Geography
Encoded Gender
Encoded Card Type


In [94]:
df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Complain,Satisfaction Score,Card Type,Point Earned,Fraud,RiskScore,BalancePerProduct,AgeRisk,HighValueCustomer,LowCreditRisk,MarketingScore,ComplainFlag,LowSatisfaction,OperationalRiskScore
0,619,0,0,42,2,0.00,1,1,1,101348.88,1,1,2,0,464,0,1,0.000000,0,0,0,0,1,1,2
1,608,2,0,41,1,83807.86,1,0,1,112542.58,0,1,3,0,456,0,1,41903.930000,0,0,0,1,1,0,1
2,502,0,0,42,8,159660.80,3,1,0,113931.57,1,1,3,0,377,0,3,39915.200000,0,1,0,1,1,0,2
3,699,0,0,39,1,0.00,2,0,0,93826.63,0,0,5,1,350,0,1,0.000000,0,0,0,0,0,0,1
4,850,2,0,43,2,125510.82,1,1,1,79084.10,0,0,5,1,425,0,1,62755.410000,0,1,0,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,0,1,39,5,0.00,2,1,0,96270.64,0,0,1,0,300,0,1,0.000000,0,0,0,0,0,1,2
9996,516,0,1,35,10,57369.61,1,1,1,101699.77,0,0,5,2,771,0,0,28684.805000,0,0,0,0,0,0,0
9997,709,0,0,36,7,0.00,1,0,1,42085.58,1,1,3,3,564,0,1,0.000000,0,0,0,0,1,0,1
9998,772,1,1,42,3,75075.31,2,1,0,92888.52,1,1,2,1,339,1,2,25025.103333,0,0,0,1,1,1,3


## Define features for X and y -> (Fraud detection - Market Risk - Operation Risk)

In [95]:
FEATURES = [col for col in df.columns if col not in ["Fraud", "MarketingScore", "OperationalRiskScore"]]
TARGET_FRAUD = "Fraud"
TARGET_MARKETING = "MarketingScore"
TARGET_OPERATIONAL = "OperationalRiskScore"

# Define X and y for each target
X = df[FEATURES]
y_fraud = df[TARGET_FRAUD]
y_marketing = df[TARGET_MARKETING]
y_operational = df[TARGET_OPERATIONAL]

from sklearn.model_selection import train_test_split
# Clean code for splitting data into train and test sets for each target variable
def split_data(X, y, test_size=0.2, random_state=42, stratify=None):
    return train_test_split(X, y, test_size=test_size, random_state=random_state, stratify=stratify)

X_train_fraud, X_test_fraud, y_train_fraud, y_test_fraud = split_data(X, y_fraud, stratify=y_fraud) # Split data for fraud prediction
X_train_marketing, X_test_marketing, y_train_marketing, y_test_marketing = split_data(X, y_marketing, stratify=y_marketing) # Split data for marketing score prediction
X_train_operational, X_test_operational, y_train_operational, y_test_operational = split_data(X, y_operational, stratify=y_operational) # Split data for operational risk score prediction

## Data scaling

In [96]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
# Scale features for fraud detection
X_train_fraud_scaled = scaler.fit_transform(X_train_fraud)
X_test_fraud_scaled = scaler.transform(X_test_fraud)
# Scale features for marketing score prediction
X_train_marketing_scaled = scaler.fit_transform(X_train_marketing)
X_test_marketing_scaled = scaler.transform(X_test_marketing)
# Scale features for operational risk score prediction
X_train_operational_scaled = scaler.fit_transform(X_train_operational)
X_test_operational_scaled = scaler.transform(X_test_operational)

print(f"Data scaled for fraud detection: {X_train_fraud_scaled.shape}, {X_test_fraud_scaled.shape}")
print(f"Data scaled for marketing score prediction: {X_train_marketing_scaled.shape}, {X_test_marketing_scaled.shape}")
print(f"Data scaled for operational risk score prediction: {X_train_operational_scaled.shape}, {X_test_operational_scaled.shape}")

Data scaled for fraud detection: (8000, 22), (2000, 22)
Data scaled for marketing score prediction: (8000, 22), (2000, 22)
Data scaled for operational risk score prediction: (8000, 22), (2000, 22)


## Data SMOTE

In [97]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42, k_neighbors=3)
# Apply SMOTE to the fraud detection training data
X_train_fraud_smote, y_train_fraud_smote = smote.fit_resample(X_train_fraud_scaled, y_train_fraud)

print(f"Original fraud training set shape: {X_train_fraud_scaled.shape}, {y_train_fraud.shape}")
print(f"SMOTE fraud training set shape: {X_train_fraud_smote.shape}, {y_train_fraud_smote.shape}")

Original fraud training set shape: (8000, 22), (8000,)
SMOTE fraud training set shape: (15468, 22), (15468,)


## Machine learning models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    accuracy_score, precision_score, recall_score, f1_score, make_scorer
)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ── Define models ────────────────────────────────────────────────────────────
model = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    "Decision Tree":       DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    "Random Forest":       RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1),
    "XGBoost":             XGBClassifier(random_state=42, eval_metric='logloss', verbosity=0, scale_pos_weight=3),
    "KNN":                 KNeighborsClassifier()
}

# ── Hyperparameter grids ─────────────────────────────────────────────────────
param_grid = {
    "Logistic Regression": {"C": [0.001, 0.01, 0.1, 1, 10], "penalty": ["l2"], "solver": ["lbfgs"]},
    "Decision Tree":       {"max_depth": [5, 10, 15, 20], "min_samples_split": [5, 10, 20], "min_samples_leaf": [2, 4, 8]},
    "Random Forest":       {"n_estimators": [100, 200, 300], "max_depth": [5, 10, 15], "min_samples_split": [5, 10], "min_samples_leaf": [2, 4]},
    "XGBoost":             {"n_estimators": [100, 200], "max_depth": [3, 5, 7], "learning_rate": [0.01, 0.05, 0.1], "subsample": [0.7, 0.8, 1.0]},
    "KNN":                 {"n_neighbors": [3, 5, 7, 9], "weights": ["uniform", "distance"]}
}

# ── Tuning helpers ───────────────────────────────────────────────────────────
def tune_model_binary(model, param_grid, X_train, y_train):
    rs = RandomizedSearchCV(model, param_grid, n_iter=10, cv=5, verbose=0,
                            random_state=42, n_jobs=-1, scoring='f1')
    rs.fit(X_train, y_train)
    return rs.best_estimator_

def tune_model_multiclass(model, param_grid, X_train, y_train):
    scorer = make_scorer(f1_score, average='weighted', zero_division=0)
    rs = RandomizedSearchCV(model, param_grid, n_iter=10, cv=5, verbose=0,
                            random_state=42, n_jobs=-1, scoring=scorer)
    rs.fit(X_train, y_train)
    return rs.best_estimator_

def get_average(y_true):
    return 'binary' if len(set(y_true)) == 2 else 'weighted'

# ── Confusion matrix plotter ─────────────────────────────────────────────────
def plot_confusion_matrices(cms: dict, model_name: str):
    """
    cms  : {'Fraud': (cm, labels), 'Marketing': (cm, labels), 'Operational': (cm, labels)}
    Saves one figure with 3 side-by-side heatmaps per model.
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Confusion Matrices — {model_name}", fontsize=15, fontweight='bold', y=1.02)

    palette = {"Fraud": "Reds", "Marketing": "Blues", "Operational": "Greens"}

    for ax, (task, (cm, labels)) in zip(axes, cms.items()):
        sns.heatmap(
            cm, annot=True, fmt='d', cmap=palette[task],
            xticklabels=labels, yticklabels=labels,
            linewidths=0.5, linecolor='grey', ax=ax
        )
        ax.set_title(task, fontsize=13, fontweight='bold')
        ax.set_xlabel("Predicted Label", fontsize=10)
        ax.set_ylabel("True Label", fontsize=10)
        ax.tick_params(axis='x', rotation=45)
        ax.tick_params(axis='y', rotation=0)

    plt.tight_layout()
    safe_name = model_name.replace(" ", "_")
    plt.show()
    print(f"  ✔ Figure saved → confusion_matrix_{safe_name}.png")

# ── Storage ──────────────────────────────────────────────────────────────────
all_results = {}
all_models  = {}

# ── Main training loop ───────────────────────────────────────────────────────
for name, clf in model.items():
    print(f"\n{'='*70}")
    print(f"  Training: {name}")
    print(f"{'='*70}")

    results = {}
    models  = {}
    cms     = {}          # will hold confusion matrices for the plot

    # ── FRAUD DETECTION (Binary) ─────────────────────────────────────────────
    print(f"\n--- Fraud Detection (Binary) ---")
    best_clf_fraud = tune_model_binary(clf, param_grid[name],
                                       X_train_fraud_smote, y_train_fraud_smote)
    y_pred_proba_fraud = best_clf_fraud.predict_proba(X_test_fraud_scaled)[:, 1]
    avg_fraud = get_average(y_test_fraud)

    # Threshold search
    thresholds   = np.arange(0.1, 0.95, 0.05)
    best_threshold, best_f1 = 0.5, 0

    for thr in thresholds:
        y_tmp = (y_pred_proba_fraud >= thr).astype(int)
        f1    = f1_score(y_test_fraud, y_tmp, average=avg_fraud, zero_division=0)
        if f1 > best_f1:
            best_f1, best_threshold = f1, thr

    y_pred_fraud = (y_pred_proba_fraud >= best_threshold).astype(int)

    cm_fraud      = confusion_matrix(y_test_fraud, y_pred_fraud)
    fraud_labels  = ["Legitimate", "Fraud"]
    cms["Fraud"]  = (cm_fraud, fraud_labels)

    results['Fraud'] = {
        "Accuracy":       accuracy_score(y_test_fraud, y_pred_fraud),
        "Precision":      precision_score(y_test_fraud, y_pred_fraud, average=avg_fraud, zero_division=0),
        "Recall":         recall_score(y_test_fraud, y_pred_fraud, average=avg_fraud, zero_division=0),
        "F1 Score":       f1_score(y_test_fraud, y_pred_fraud, average=avg_fraud, zero_division=0),
        "ROC AUC":        roc_auc_score(y_test_fraud, y_pred_proba_fraud),
        "Best Threshold": round(best_threshold, 2)
    }
    models['Fraud'] = best_clf_fraud

    print(f"  Best Threshold : {best_threshold:.2f}")
    print(f"  Accuracy       : {results['Fraud']['Accuracy']:.4f}")
    print(f"  Precision      : {results['Fraud']['Precision']:.4f}")
    print(f"  Recall         : {results['Fraud']['Recall']:.4f}")
    print(f"  F1 Score       : {results['Fraud']['F1 Score']:.4f}")
    print(f"  ROC AUC        : {results['Fraud']['ROC AUC']:.4f}")
    print(f"\n  Confusion Matrix (Fraud):\n{cm_fraud}")

    # ── MARKETING SCORE PREDICTION (Multiclass) ──────────────────────────────
    print(f"\n--- Marketing Score Prediction (Multiclass) ---")
    best_clf_marketing = tune_model_multiclass(clf, param_grid[name],
                                               X_train_marketing_scaled, y_train_marketing)
    y_pred_marketing   = best_clf_marketing.predict(X_test_marketing_scaled)
    avg_marketing      = get_average(y_test_marketing)

    cm_marketing          = confusion_matrix(y_test_marketing, y_pred_marketing)
    marketing_labels      = sorted(set(y_test_marketing))          # e.g. [0, 1, 2, 3]
    cms["Marketing"]      = (cm_marketing, marketing_labels)

    results['Marketing'] = {
        "Accuracy":  accuracy_score(y_test_marketing, y_pred_marketing),
        "Precision": precision_score(y_test_marketing, y_pred_marketing, average=avg_marketing, zero_division=0),
        "Recall":    recall_score(y_test_marketing, y_pred_marketing, average=avg_marketing, zero_division=0),
        "F1 Score":  f1_score(y_test_marketing, y_pred_marketing, average=avg_marketing, zero_division=0)
    }
    models['Marketing'] = best_clf_marketing

    print(f"  Accuracy  : {results['Marketing']['Accuracy']:.4f}")
    print(f"  Precision : {results['Marketing']['Precision']:.4f}")
    print(f"  Recall    : {results['Marketing']['Recall']:.4f}")
    print(f"  F1 Score  : {results['Marketing']['F1 Score']:.4f}")
    print(f"\n  Confusion Matrix (Marketing):\n{cm_marketing}")

    # ── OPERATIONAL RISK SCORE PREDICTION (Multiclass) ───────────────────────
    print(f"\n--- Operational Risk Score Prediction (Multiclass) ---")
    best_clf_operational = tune_model_multiclass(clf, param_grid[name],
                                                 X_train_operational_scaled, y_train_operational)
    y_pred_operational   = best_clf_operational.predict(X_test_operational_scaled)
    avg_operational      = get_average(y_test_operational)

    cm_operational          = confusion_matrix(y_test_operational, y_pred_operational)
    operational_labels      = sorted(set(y_test_operational))
    cms["Operational"]      = (cm_operational, operational_labels)

    results['Operational'] = {
        "Accuracy":  accuracy_score(y_test_operational, y_pred_operational),
        "Precision": precision_score(y_test_operational, y_pred_operational, average=avg_operational, zero_division=0),
        "Recall":    recall_score(y_test_operational, y_pred_operational, average=avg_operational, zero_division=0),
        "F1 Score":  f1_score(y_test_operational, y_pred_operational, average=avg_operational, zero_division=0)
    }
    models['Operational'] = best_clf_operational

    print(f"  Accuracy  : {results['Operational']['Accuracy']:.4f}")
    print(f"  Precision : {results['Operational']['Precision']:.4f}")
    print(f"  Recall    : {results['Operational']['Recall']:.4f}")
    print(f"  F1 Score  : {results['Operational']['F1 Score']:.4f}")
    print(f"\n  Confusion Matrix (Operational):\n{cm_operational}")

    # ── Plot all 3 confusion matrices for this model ─────────────────────────
    plot_confusion_matrices(cms, name)

    all_results[name] = results
    all_models[name]  = models

print(f"\n{'='*70}")
print("  MODEL TRAINING COMPLETE")
print(f"{'='*70}")


  Training: Logistic Regression

--- Fraud Detection (Binary) ---


## Save Results and Models

In [ ]:
import json
import pickle
import os

# Create results directory
results_dir = "fraud_detection_results"
os.makedirs(results_dir, exist_ok=True)

# Create models subdirectory
models_dir = os.path.join(results_dir, "models")
os.makedirs(models_dir, exist_ok=True)

# Convert results to DataFrame for better visualization
results_summary = {}

for model_name, target_results in all_results.items():
    for target_name, metrics in target_results.items():
        key = f"{model_name}_{target_name}"
        results_summary[key] = metrics

results_df = pd.DataFrame(results_summary).T
print("\n" + "="*100)
print("COMPREHENSIVE MODEL RESULTS SUMMARY")
print("="*100)
print(results_df.to_string())

# Save to CSV
results_df.to_csv(f"{results_dir}/all_results.csv")
print(f"\n✓ Results saved to: {results_dir}/all_results.csv")

# Find best models for each task
print("\n" + "="*80)
print("BEST MODELS BY TASK")
print("="*80)

best_models_info = {}

for target in ['Fraud', 'Marketing', 'Operational']:
    target_results = {}
    for model_name in all_results.keys():
        if target in all_results[model_name]:
            target_results[model_name] = all_results[model_name][target]
    
    if target_results:
        best_model = max(target_results.items(), key=lambda x: x[1]['F1 Score'])
        best_models_info[target] = best_model[0]
        
        print(f"\n{target} Detection:")
        print(f"  Best Model: {best_model[0]}")
        print(f"  Metrics: {best_model[1]}")
        
        # Save best model metrics
        with open(f"{results_dir}/best_{target.lower()}_metrics.json", 'w') as f:
            metrics_dict = {k: float(v) if isinstance(v, (float, np.floating, np.integer)) else v 
                          for k, v in best_model[1].items()}
            json.dump({"Model": best_model[0], "Metrics": metrics_dict}, f, indent=4)

# ────────────────────────────────────────────────────────────────────────────
# SAVE ALL TRAINED MODELS
# ────────────────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("SAVING ALL TRAINED MODELS")
print("="*80)

model_registry = {}

for model_name, model_dict in all_models.items():
    print(f"\n{model_name}:")
    model_registry[model_name] = {}
    
    for task_name, trained_model in model_dict.items():
        # Create safe filename
        safe_model_name = model_name.replace(" ", "_")
        safe_task_name = task_name.replace(" ", "_")
        model_filename = f"{safe_model_name}_{safe_task_name}.pkl"
        model_filepath = os.path.join(models_dir, model_filename)
        
        # Save model using pickle
        with open(model_filepath, 'wb') as f:
            pickle.dump(trained_model, f)
        
        model_registry[model_name][task_name] = {
            "filename": model_filename,
            "filepath": model_filepath,
            "task": task_name,
            "model_type": type(trained_model).__name__
        }
        
        print(f"  ✓ {task_name:25s} → {model_filename}")

# Save model registry (JSON)
registry_filepath = os.path.join(results_dir, "model_registry.json")
with open(registry_filepath, 'w') as f:
    json.dump(model_registry, f, indent=4)
print(f"\n✓ Model registry saved to: {registry_filepath}")

# Create detailed fraud detection analysis
print("\n" + "="*80)
print("FRAUD DETECTION - DETAILED ANALYSIS")
print("="*80)

fraud_analysis = {}
for model_name, results in all_results.items():
    if 'Fraud' in results:
        fraud_analysis[model_name] = results['Fraud']

fraud_df = pd.DataFrame(fraud_analysis).T
fraud_df = fraud_df.sort_values('F1 Score', ascending=False)

print("\nRanked by F1 Score (Best Balance of Precision & Recall):")
print(fraud_df.to_string())

fraud_df.to_csv(f"{results_dir}/fraud_detection_ranking.csv")
print(f"\n✓ Fraud detection analysis saved to: {results_dir}/fraud_detection_ranking.csv")

# Create comparison summary
print("\n" + "="*80)
print("METRIC COMPARISON SUMMARY")
print("="*80)

comparison_data = []
for model_name in all_results.keys():
    for target in ['Fraud', 'Marketing', 'Operational']:
        if target in all_results[model_name]:
            metrics = all_results[model_name][target]
            comparison_data.append({
                'Model': model_name,
                'Task': target,
                'Accuracy': metrics.get('Accuracy', 0),
                'Precision': metrics.get('Precision', 0),
                'Recall': metrics.get('Recall', 0),
                'F1 Score': metrics.get('F1 Score', 0),
                'ROC AUC': metrics.get('ROC AUC', 'N/A')
            })

comparison_df = pd.DataFrame(comparison_data)
comparison_df.to_csv(f"{results_dir}/detailed_comparison.csv", index=False)

print("\nDetailed comparison saved!")
print(comparison_df.to_string(index=False))

# Save complete results as JSON
with open(f"{results_dir}/complete_results.json", 'w') as f:
    all_results_serializable = {}
    for model_name, targets in all_results.items():
        all_results_serializable[model_name] = {}
        for target_name, metrics in targets.items():
            all_results_serializable[model_name][target_name] = {
                k: float(v) if isinstance(v, (np.floating, np.integer)) else v 
                for k, v in metrics.items()
            }
    json.dump(all_results_serializable, f, indent=4)

print(f"\n✓ Complete results saved to: {results_dir}/complete_results.json")

# ────────────────────────────────────────────────────────────────────────────
# SUMMARY
# ────────────────────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("SUMMARY: ALL FILES AND MODELS SAVED")
print("="*80)

print(f"\n📁 Main Directory: {results_dir}/")
print(f"📁 Models Directory: {models_dir}/")

print("\n📊 Results Files:")
print(f"  ✓ all_results.csv")
print(f"  ✓ fraud_detection_ranking.csv")
print(f"  ✓ detailed_comparison.csv")
print(f"  ✓ complete_results.json")
print(f"  ✓ best_fraud_metrics.json")
print(f"  ✓ best_marketing_metrics.json")
print(f"  ✓ best_operational_metrics.json")
print(f"  ✓ model_registry.json")

print("\n🤖 Trained Models (15 total):")
for model_name in all_models.keys():
    print(f"\n  {model_name}:")
    for task in ['Fraud', 'Marketing', 'Operational']:
        safe_name = model_name.replace(" ", "_")
        print(f"    ✓ {safe_name}_{task}.pkl")

print("\n✨ Quick Access:")
print(f"  • Best Fraud Model     : {best_models_info.get('Fraud', 'N/A')}")
print(f"  • Best Marketing Model : {best_models_info.get('Marketing', 'N/A')}")
print(f"  • Best Operational Model: {best_models_info.get('Operational', 'N/A')}")

print("\n💡 To load a model later:")
print(f"""
  import pickle
  with open('{models_dir}/model_name.pkl', 'rb') as f:
      model = pickle.load(f)
  predictions = model.predict(X_test)
""")